# Clase 15 — SOLUCIONES DEL INSTRUCTOR

Este notebook contiene las soluciones de todos los ejercicios.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, confusion_matrix)
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
print("Listo.")

## Ejercicio 1: Explora el dataset Iris

In [ ]:
iris = sns.load_dataset("iris")
iris.head()

In [ ]:
iris["species"].value_counts()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=iris, x="species", palette="Set2")
plt.title("Distribucion de especies de Iris")
plt.xlabel("Especie")
plt.ylabel("Cantidad")
plt.savefig("iris_countplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Otro tipo de grafico
plt.figure(figsize=(6, 4))
sns.scatterplot(data=iris, x="sepal_length", y="petal_length",
                hue="species", palette="Dark2")
plt.title("Largo de sepalo vs petalo")
plt.show()

## Ejercicio 2: Boxplot de Iris

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=iris, x="species", y="petal_length", palette="pastel")
plt.title("Largo de petalo por especie")
plt.xlabel("Especie")
plt.ylabel("Largo del petalo (cm)")
plt.show()

## Ejercicio 3: Explorar el target de Attrition

In [ ]:
url = ("https://raw.githubusercontent.com/IBM/"
       "employee-attrition-aif360/master/"
       "data/emp_attrition.csv")
df = pd.read_csv(url)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Attrition", palette=["#6B1525", "#C82B40"])
plt.title("Distribucion de Attrition")
plt.xlabel("Renuncio")
plt.ylabel("Cantidad de empleados")
plt.show()

pct = (df["Attrition"] == "Yes").mean() * 100
print(f"Porcentaje que renuncia: {pct:.1f}% — NO esta balanceado.")

## Ejercicio 4: Encoding

In [ ]:
df["Attrition"] = (df["Attrition"] == "Yes").astype(int)

cols = ["Age", "MonthlyIncome", "DistanceFromHome",
        "YearsAtCompany", "TotalWorkingYears",
        "OverTime", "JobSatisfaction", "MaritalStatus",
        "EnvironmentSatisfaction", "NumCompaniesWorked"]
df2 = df[cols + ["Attrition"]].copy()

# Las categoricas son OverTime y MaritalStatus
df_enc = pd.get_dummies(df2, columns=["OverTime", "MaritalStatus"],
                        drop_first=True, dtype=int)
print(f"Features: {df_enc.shape[1] - 1}")
df_enc.head(3)

## Setup para ejercicios 5-7

In [ ]:
X = df_enc.drop(columns=["Attrition"])
y = df_enc["Attrition"]

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_s, y)
y_pred = modelo.predict(X_s)

## Ejercicio 5: Compara Base vs Balanced

In [ ]:
modelo_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo_bal.fit(X_s, y)
y_pred_bal = modelo_bal.predict(X_s)

print("=== Base ===")
print(f"  Precision: {precision_score(y, y_pred):.3f}")
print(f"  Recall:    {recall_score(y, y_pred):.3f}")
print()
print("=== Balanced ===")
print(f"  Precision: {precision_score(y, y_pred_bal):.3f}")
print(f"  Recall:    {recall_score(y, y_pred_bal):.3f}")

In [ ]:
cm_bal = confusion_matrix(y, y_pred_bal)
cm_base = confusion_matrix(y, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_bal, annot=True, fmt="d", cmap="Reds")
plt.ylabel("Real")
plt.xlabel("Prediccion")
plt.title("Confusion - Modelo Balanced")
plt.show()

print(f"VP base: {cm_base[1,1]}, VP balanced: {cm_bal[1,1]}")
print(f"Diferencia: +{cm_bal[1,1] - cm_base[1,1]} empleados detectados")

## Ejercicio 6: SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_s, y)

m_smote = LogisticRegression(max_iter=1000)
m_smote.fit(X_smote, y_smote)
y_pred_smote = m_smote.predict(X_s)

print(f"Precision: {precision_score(y, y_pred_smote):.3f}")
print(f"Recall:    {recall_score(y, y_pred_smote):.3f}")

cm_smote = confusion_matrix(y, y_pred_smote)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_smote, annot=True, fmt="d", cmap="Reds")
plt.ylabel("Real")
plt.xlabel("Prediccion")
plt.title("Confusion - SMOTE")
plt.savefig("smote_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nVP base: {cm_base[1,1]}, VP SMOTE: {cm_smote[1,1]}")
print(f"Diferencia: +{cm_smote[1,1] - cm_base[1,1]} empleados detectados")

## Ejercicio 7: Bank Marketing

In [ ]:
# Cargar
URL_BANK = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-14/bank.csv"
df_bank = pd.read_csv(URL_BANK)
print(f"{df_bank.shape[0]} clientes, acepta: {df_bank['response'].mean()*100:.1f}%")

In [ ]:
# Preparar (mismo proceso que clase 14)
cols_bank = ["age", "job", "marital", "education", "default", "balance",
             "housing", "loan", "contact", "duration", "campaign",
             "pdays", "previous", "poutcome", "response"]
df_b = df_bank[cols_bank].copy()

df_b_enc = pd.get_dummies(df_b,
    columns=["job", "marital", "education", "default",
             "housing", "loan", "contact", "poutcome"],
    drop_first=True, dtype=int)

X_b = df_b_enc.drop(columns=["response"])
y_b = df_b_enc["response"]

scaler_b = StandardScaler()
X_b_s = scaler_b.fit_transform(X_b)

In [ ]:
# Modelo base
m_b_base = LogisticRegression(max_iter=1000)
m_b_base.fit(X_b_s, y_b)
p_b_base = m_b_base.predict(X_b_s)

# Modelo balanced
m_b_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
m_b_bal.fit(X_b_s, y_b)
p_b_bal = m_b_bal.predict(X_b_s)

print("=== Bank Marketing ===")
print("\nModelo Base:")
print(f"  Precision: {precision_score(y_b, p_b_base):.3f}")
print(f"  Recall:    {recall_score(y_b, p_b_base):.3f}")
cm_b_base = confusion_matrix(y_b, p_b_base)
print(f"  VP: {cm_b_base[1,1]}, FN: {cm_b_base[1,0]}")

print("\nModelo Balanced:")
print(f"  Precision: {precision_score(y_b, p_b_bal):.3f}")
print(f"  Recall:    {recall_score(y_b, p_b_bal):.3f}")
cm_b_bal = confusion_matrix(y_b, p_b_bal)
print(f"  VP: {cm_b_bal[1,1]}, FN: {cm_b_bal[1,0]}")

print(f"\nDiferencia: +{cm_b_bal[1,1] - cm_b_base[1,1]} clientes detectados")